# Phase 2 — Data Cleaning & Database Load
## UCI Online Retail II — Customer Intelligence Project

**Goal of this phase:** turn the raw, messy transaction file into clean, analysis-ready tables, then load them into PostgreSQL.

**Approach (decided in Phase 1):**
- Clean **sequentially**, logging the cumulative row count after every step so data loss is fully traceable. The Phase 1 issues overlap, so they cannot be removed independently.
- Separate **non-product stock codes** (postage, manual adjustments, bad debt, etc.) by their code — not by clipping numeric outliers — so legitimate bulk orders are preserved.
- Produce **two cleaned tables** plus a returns table:
  - `transactions_clean` — all valid sales including guest checkouts -> for revenue & market basket analysis
  - `customers_clean` — identifiable customers only -> for RFM, CLV (BG/NBD) & cohort analysis
  - `returns` — cancellations, kept separately for returns analysis
- Keep **all 43 countries** but add a `region` flag (UK / Non-UK).
- Load all three tables into **PostgreSQL**.


## 1. Imports & Settings

In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

# Locate the repo root portably by walking up from the current working
# directory until the 'data' folder is found, so the notebook runs on any
# machine and from any subfolder.
def find_project_root(marker='data'):
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / marker).is_dir():
            return parent
    raise FileNotFoundError(
        f"Could not find project root: no '{marker}/' folder above {Path.cwd()}. "
        "Run this notebook from inside the cloned repository."
    )

PROJECT_ROOT   = find_project_root()
RAW_PATH       = str(PROJECT_ROOT / 'data' / 'raw' / 'online_retail_II.xlsx')
PROCESSED_PATH = str(PROJECT_ROOT / 'data' / 'processed') + os.sep
os.makedirs(PROCESSED_PATH, exist_ok=True)

## 2. Load Raw Data

We reload from the original Excel file so this notebook runs independently of Phase 1.
Both yearly sheets are concatenated into one dataframe.

In [2]:
# Read every sheet in the workbook and stack them into a single dataframe
xls = pd.ExcelFile(RAW_PATH)
df = pd.concat([pd.read_excel(RAW_PATH, sheet_name=s) for s in xls.sheet_names],
               ignore_index=True)

# Standardise column names to snake_case so later code is clean and consistent
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

print(f'Raw combined data: {len(df):,} rows')
df.head(3)

Raw combined data: 1,067,371 rows


,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom


In [3]:
# Reproducibility check: confirm the exact raw file is present and record its
# checksum, so a reviewer can verify they started from the identical input.
import hashlib

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

print(f'Raw data checksum (SHA-256, first 12): online_retail_II.xlsx  {sha256(RAW_PATH)[:12]}')

Raw data checksum (SHA-256, first 12): online_retail_II.xlsx  bcbe73b35f5b


## 3. Row-Count Logger

A small helper that records how many rows remain after each cleaning step.
This produces a transparent audit trail of exactly where data was removed.

In [4]:
# Each call appends a labelled checkpoint and prints the running total
cleaning_log = []

def log_step(step_name, dataframe):
    cleaning_log.append({'step': step_name, 'rows_remaining': len(dataframe)})
    print(f'{step_name:48s} {len(dataframe):>10,} rows')

# Starting point — the raw data before any cleaning
log_step('0. Raw combined data', df)

0. Raw combined data                              1,067,371 rows


## 4. Step 1 — Remove Exact Duplicate Rows

34,335 rows (3.22%) are exact duplicates across every column. These would double-count
sales, so we drop them first, before any splitting.

In [5]:
# Removing fully identical rows; keeps the first occurrence of each
df = df.drop_duplicates().reset_index(drop=True)
log_step('1. After removing exact duplicates', df)

1. After removing exact duplicates                1,033,036 rows


## 5. Step 2 — Separate Cancellations (Returns)

Invoices whose number starts with **'C'** are cancellations / returns. They carry negative
quantities and represent reversed sales. We move them to a separate `returns` table rather
than deleting them — returns are useful for their own analysis — and continue cleaning the
remaining genuine sales.

In [6]:
# Flag any invoice whose number begins with the letter 'C'
is_cancellation = df['invoice'].astype(str).str.startswith('C')

# Set aside the cancellations for later returns analysis
returns = df[is_cancellation].copy()

# Keep only the non-cancellation rows in the main pipeline
df = df[~is_cancellation].reset_index(drop=True)

print(f'Returns set aside: {len(returns):,} rows')
log_step('2. After separating cancellations', df)

Returns set aside: 19,104 rows
2. After separating cancellations                 1,013,932 rows


## 6. Step 3 — Remove Non-Product Stock Codes

Some stock codes are not products — postage (POST, DOT), carriage (C2/C3), manual
adjustments (M, ADJUST), discounts (D), samples (S), bad-debt entries (B), bank charges,
Amazon fees, gift cards, and similar administrative entries. These are the source of the
extreme price/quantity values seen in Phase 1.



In [7]:
# Known non-product / administrative stock codes in this dataset.
# We remove ONLY these (plus gift-card codes) and keep everything else,
# so genuine products with non-standard codes are never dropped by accident.
NON_PRODUCT_CODES = {
    'POST', 'DOT', 'C2', 'C3',      # postage & carriage
    'M', 'm', 'D', 'S', 'B',        # manual, discount, sample, bad-debt adjustments
    'BANK CHARGES', 'AMAZONFEE',    # fees
    'ADJUST', 'ADJUST2', 'CRUK',    # adjustments & commission
    'PADS', 'TEST001', 'TEST002',   # filler & test entries
    'GIFT',                         # standalone gift code (sibling of gift_ vouchers)
}

# Normalise stock codes to clean strings for matching
codes = df['stockcode'].astype(str).str.strip()

# Flag a row as non-product if its code is in the blocklist or is a gift-card code
is_non_product = (
    codes.str.upper().isin({c.upper() for c in NON_PRODUCT_CODES})
    | codes.str.lower().str.startswith('gift_')
)

# Show exactly what will be removed, for transparency
print('Stock codes flagged for removal:')
removed_summary = (df[is_non_product].groupby('stockcode')
                     .agg(rows=('stockcode', 'size'),
                          example_description=('description',
                              lambda s: s.dropna().iloc[0] if s.notna().any() else ''))
                     .sort_values('rows', ascending=False))
display(removed_summary)

Stock codes flagged for removal:


,rows,example_description
stockcode,,
POST,1858,POSTAGE
DOT,1422,DOTCOM POSTAGE
M,853,Manual
C2,270,CARRIAGE
ADJUST,36,Adjustment by john on 26/01/2010 16
BANK CHARGES,34,Bank Charges
gift_0001_30,29,Dotcomgiftshop Gift Voucher £30.00
gift_0001_20,29,Dotcomgiftshop Gift Voucher £20.00
PADS,18,PADS TO MATCH ALL CUSHIONS


In [8]:
# Remove the flagged administrative rows; keep all genuine products
df = df[~is_non_product].reset_index(drop=True)
log_step('3. After removing non-product codes', df)

# VERIFICATION — list any kept codes that are not the standard 5-digit format.
# These should now all be genuine products (e.g. the DCGS series). If anything
# here looks administrative, add it to NON_PRODUCT_CODES above and rerun.
kept_nonstandard = df[~df['stockcode'].astype(str).str.match(r'^\d{5}')]
print(f"Remaining non-standard codes kept (should all be real products): "
      f"{kept_nonstandard['stockcode'].nunique()} distinct codes")
display(kept_nonstandard.groupby('stockcode')
          .agg(rows=('stockcode', 'size'),
               example_description=('description',
                   lambda s: s.dropna().iloc[0] if s.notna().any() else ''))
          .sort_values('rows', ascending=False)
          .head(25))

3. After removing non-product codes               1,009,302 rows
Remaining non-standard codes kept (should all be real products): 35 distinct codes


,rows,example_description
stockcode,,
DCGS0058,31,MISO PRETTY GUM
DCGSSGIRL,25,update
DCGSSBOY,23,update
DCGS0003,14,BOXED GLASS ASHTRAY
DCGS0076,14,SUNJAR LED NIGHT NIGHT LIGHT
DCGS0069,6,OOH LA LA DOGS COLLAR
DCGS0004,5,HAYNES CAMPER SHOULDER BAG
DCGS0072,4,CAT CAMOUFLAGUE COLLAR
DCGS0066N,4,NAVY CUDDLES DOG HOODIE


## 7. Step 4 — Remove Invalid Quantity and Price

After cancellations are gone, any remaining rows with **quantity <= 0** are manual
adjustments/write-offs, and rows with **price <= 0** are free items or pricing errors.
Neither represents a genuine paid sale, so both are removed.

In [9]:
# Remove non-positive quantities (adjustments / write-offs that were not formal cancellations)
df = df[df['quantity'] > 0].reset_index(drop=True)
log_step('4a. After removing quantity <= 0', df)

# Remove non-positive prices (free items, zero/negative pricing errors)
df = df[df['price'] > 0].reset_index(drop=True)
log_step('4b. After removing price <= 0', df)

4a. After removing quantity <= 0                  1,005,911 rows
4b. After removing price <= 0                     1,003,340 rows


## 8. Step 5 — Add Derived Columns

We add a `revenue` column (quantity x price) and a `region` flag (UK vs Non-UK).
The region flag lets us analyse the dominant UK market cleanly while still keeping a global view.

In [10]:
# Line-level revenue, needed for nearly all downstream analysis
df['revenue'] = df['quantity'] * df['price']

# Region flag — United Kingdom vs everywhere else
df['region'] = np.where(df['country'] == 'United Kingdom', 'UK', 'Non-UK')

df[['invoice', 'stockcode', 'quantity', 'price', 'revenue', 'country', 'region']].head()

,invoice,stockcode,quantity,price,revenue,country,region
0,489434,85048,12,6.95,83.40,United Kingdom,UK
1,489434,79323P,12,6.75,81.00,United Kingdom,UK
2,489434,79323W,12,6.75,81.00,United Kingdom,UK
3,489434,22041,48,2.10,100.80,United Kingdom,UK
4,489434,21232,24,1.25,30.00,United Kingdom,UK


## 9. Build the Two Cleaned Tables

`transactions_clean` keeps **all** valid sales, including guest checkouts with no customer ID
(needed for revenue and market basket analysis). `customers_clean` keeps **only** rows with a
known customer ID (needed for RFM, CLV and cohort analysis), with the ID converted to a clean
integer.

In [11]:
# transactions_clean — every valid sale, guests included
transactions_clean = df.copy()

# customers_clean — only rows we can attribute to a specific customer
customers_clean = df[df['customer_id'].notna()].copy()

# Convert customer_id from float (forced by the NaNs) to a proper integer
customers_clean['customer_id'] = customers_clean['customer_id'].astype('int64')

log_step('5. transactions_clean (all valid sales)', transactions_clean)
log_step('6. customers_clean (identifiable only)', customers_clean)

5. transactions_clean (all valid sales)           1,003,340 rows
6. customers_clean (identifiable only)              776,579 rows


## 10. Cleaning Audit Trail

The full record of how many rows survived each step.

In [12]:
# Turn the log into a tidy table and show the cumulative effect of cleaning
audit = pd.DataFrame(cleaning_log)
audit['rows_removed'] = audit['rows_remaining'].diff().fillna(0).astype(int).abs()
audit['pct_of_raw'] = (audit['rows_remaining'] / cleaning_log[0]['rows_remaining'] * 100).round(2)
audit

,step,rows_remaining,rows_removed,pct_of_raw
0,0. Raw combined data,1067371,0,100.00
1,1. After removing exact duplicates,1033036,34335,96.78
2,2. After separating cancellations,1013932,19104,94.99
3,3. After removing non-product codes,1009302,4630,94.56
4,4a. After removing quantity <= 0,1005911,3391,94.24
5,4b. After removing price <= 0,1003340,2571,94.00
6,5. transactions_clean (all valid sales),1003340,0,94.00
7,6. customers_clean (identifiable only),776579,226761,72.76


## 11. Final Sanity Checks

Confirm the cleaned tables have no remaining invalid values before loading.

In [13]:
# These should all report 0 / False if cleaning worked correctly
print('transactions_clean checks')
print('  any quantity <= 0 :', (transactions_clean['quantity'] <= 0).any())
print('  any price    <= 0 :', (transactions_clean['price'] <= 0).any())
print('  any duplicates    :', transactions_clean.duplicated().any())
print()
print('customers_clean checks')
print('  any missing cust  :', customers_clean['customer_id'].isna().any())
print('  customer_id dtype :', customers_clean['customer_id'].dtype)
print('  unique customers  :', f"{customers_clean['customer_id'].nunique():,}")

transactions_clean checks
  any quantity <= 0 : False
  any price    <= 0 : False
  any duplicates    : False

customers_clean checks
  any missing cust  : False
  customer_id dtype : int64
  unique customers  : 5,852


## 12. Save Cleaned Files (CSV backup)

We save local CSV copies in `data/processed/` — a backup and a convenient source for Tableau later.
(These are git-ignored, so they stay off GitHub.)

In [14]:
transactions_clean.to_csv(PROCESSED_PATH + 'transactions_clean.csv', index=False)
customers_clean.to_csv(PROCESSED_PATH + 'customers_clean.csv', index=False)
returns.to_csv(PROCESSED_PATH + 'returns.csv', index=False)
print('Saved: transactions_clean.csv, customers_clean.csv, returns.csv')

Saved: transactions_clean.csv, customers_clean.csv, returns.csv


## 13. Load into PostgreSQL

We load all three tables into a PostgreSQL database using SQLAlchemy — the same connection
approach that worked in the previous project (local Mac trust authentication, no password).

**Before running:** create the database in your terminal:
```bash
psql -U $(whoami) -c "CREATE DATABASE retail_db;"
```
If that errors, try `createdb retail_db`.

In [15]:
from sqlalchemy import create_engine, text
import getpass

# Connect the same way that worked in the previous project:
# - getpass.getuser() pulls your Mac login, so the username is never hardcoded
# - no password field — local Mac PostgreSQL uses trust auth for your own user
# - future=True runs SQLAlchemy in full 2.0 mode
username = getpass.getuser()
DB_URL = f'postgresql+psycopg2://{username}@localhost:5432/retail_db'
engine = create_engine(DB_URL, future=True)

# Test the connection before loading anything
with engine.connect() as conn:
    row = conn.execute(text('SELECT current_database(), current_user')).fetchone()
    print(f'Connected to database: {row[0]}')
    print(f'Connected as user:     {row[1]}')

Connected to database: retail_db
Connected as user:     omartouzani


In [16]:
# Helper to load a dataframe into a table.
# chunksize keeps memory low; method='multi' batches inserts for speed.
# chunksize * n_columns must stay under PostgreSQL's parameter limit (65,535),
# so 5,000 rows x ~10 columns = 50,000 is safe.
def load_to_postgres(dataframe, table_name):
    dataframe.to_sql(
        table_name,
        engine,
        if_exists='replace',   # drop & recreate so reruns are clean
        index=False,
        chunksize=5000,
        method='multi'
    )
    print(f'Loaded {len(dataframe):>9,} rows -> {table_name}')

load_to_postgres(transactions_clean, 'transactions_clean')
load_to_postgres(customers_clean,    'customers_clean')
load_to_postgres(returns,            'returns')

Loaded 1,003,340 rows -> transactions_clean
Loaded   776,579 rows -> customers_clean
Loaded    19,104 rows -> returns


In [17]:
# Verify the load by counting rows directly in the database
with engine.connect() as conn:
    for tbl in ['transactions_clean', 'customers_clean', 'returns']:
        n = conn.execute(text(f'SELECT COUNT(*) FROM {tbl}')).scalar()
        print(f'{tbl:22s} {n:>10,} rows in PostgreSQL')

transactions_clean      1,003,340 rows in PostgreSQL
customers_clean           776,579 rows in PostgreSQL
returns                    19,104 rows in PostgreSQL


## 14. Phase 2 Summary

- Raw data cleaned through a transparent, sequential, logged process.
- Cancellations and non-product codes separated rather than blindly deleted.
- Two analysis-ready tables produced plus a returns table.
- All loaded into PostgreSQL, ready for Phase 3.

**Next — Phase 3: Cohort Analysis.** We build a monthly retention matrix from `customers_clean`
to see how well the business retains customers over time.
